In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from os import mkdir, listdir
from os.path import isdir, isfile
import re
from matplotlib.cm import ScalarMappable
import textwrap
import goatools

# GO Enrichment Analysis 

In [3]:
annotation_df = pd.read_csv("../data/annotation_KO_GO.csv")

In [4]:
# there are NaN values within the annotation_df
for entry in annotation_df.ID:
    if type(entry) != str:
        print(entry)

nan
nan
nan


In [5]:
annotation_df["ID"] = annotation_df.ID.apply(lambda x: x.split(".")[1].replace("T","G") if type(x) == str else x)

In [6]:
annotation_df = annotation_df.drop_duplicates(keep="first")

In [7]:
annotation_df

,ID,KO,KO_definition,Score,Second.best,Score2,UniProtKB,KEGG.ID,KO.Number,GO.terms
0,G000001,NaN,NaN,6.0,K21595,1.0,NaN,NaN,NaN,NaN
1,G000002,K07188,"LIPE, HSL; hormone-sensitive lipase [EC:3.1.1.79]",171.0,NaN,NaN,P54310,LIPS_MOUSE,mmu:16890;,caveola [GO:0005901]; cytoplasm [GO:0005737]; ...
2,G000003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,G000004,NaN,NaN,8.0,K10251,1.0,NaN,NaN,NaN,NaN
4,G000004,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
37782,G028913,NaN,NaN,52.0,NaN,NaN,NaN,NaN,NaN,NaN
37783,G028914,NaN,NaN,2.0,K01883,1.0,NaN,NaN,NaN,NaN
37784,G028915,NaN,NaN,3.0,K17604,1.0,NaN,NaN,NaN,NaN
37785,G028916,NaN,NaN,9.0,K11274,1.0,P10273,POL_FLV,vg:1724726;,host cell late endosome membrane [GO:0044185];...


In [8]:
annotation_df[annotation_df.ID == "G000001"]["GO.terms"].values[0]

nan

In [9]:
annotation_df_goterms = annotation_df[annotation_df["GO.terms"].isna() == False]
# write goatools association file
pattern = r'\[([^\]]+)\]'
with open("../data/goatools_data/associations.txt","w") as association_file:
    for gene in annotation_df_goterms.ID:
        if len(annotation_df_goterms[annotation_df_goterms.ID == gene]["GO.terms"].values) != 0:
            if type(annotation_df_goterms[annotation_df_goterms.ID == gene]["GO.terms"].values[0]) != float:
                for goterm in annotation_df_goterms[annotation_df_goterms.ID == gene]["GO.terms"].values:
                    if type(goterm) == str:
                        result = re.findall(pattern, goterm)
                        association_file.write(gene+"\t")

                        for go in result[:-1]:
                            association_file.write(go+";")
                        association_file.write(go+"\n")

In [10]:
with open("../data/goatools_data/associations.txt","r") as association_file:
    lines = association_file.readlines()
print("[*] Length of association_file: {}".format(len(lines)))

[*] Length of association_file: 11561


In [11]:
# write population file 
with open("../data/goatools_data/population.txt","w") as popfile:
    for identifier in annotation_df_goterms.ID:
        popfile.write(identifier+"\n")

In [12]:
unique_celltypes = pd.read_table("../results/processedData/normalized_mean/unique_celltypes.table", sep=";")

In [13]:
unique_celltypes["ID"] = unique_celltypes.ID.apply(lambda x: x.split("-")[1] if type(x) == str else x)

In [14]:
def construct_celltype_sample_data_for_goatools(unique_celltypes:pd.DataFrame, savep:str, population_table:pd.DataFrame)->int:
    for col in unique_celltypes.columns:
        if col != "ID":
            print("[*] Working with {}".format(col))
            id_to_celltype = unique_celltypes[unique_celltypes[col] == 1.0]
            print("\t[*] Number of unique genes for celltype: {}".format(len(id_to_celltype)))
            with open(savep+"sample_"+col+".txt", "w") as samplefile:
                ident = []
                for identifier in id_to_celltype.ID:
                    if identifier in list(population_table.ID):
                        samplefile.write(identifier+"\n")
                        ident.append(identifier)
                print("\t[*] Remaining IDs for GO analysis: {}".format(len(ident)))
    return 0

In [15]:
pop = pd.read_table("../data/goatools_data/population.txt", header=None)
pop.columns = ["ID"]

In [16]:
# write sample files
construct_celltype_sample_data_for_goatools(unique_celltypes, "../data/goatools_data/", pop)

[*] Working with Ec_Head
	[*] Number of unique genes for celltype: 29
	[*] Remaining IDs for GO analysis: 16
[*] Working with En_Head
	[*] Number of unique genes for celltype: 9
	[*] Remaining IDs for GO analysis: 6
[*] Working with Ec_BodyCol.SC
	[*] Number of unique genes for celltype: 2
	[*] Remaining IDs for GO analysis: 1
[*] Working with I_ISC
	[*] Number of unique genes for celltype: 1
	[*] Remaining IDs for GO analysis: 0
[*] Working with Ec_Peduncle
	[*] Number of unique genes for celltype: 22
	[*] Remaining IDs for GO analysis: 9
[*] Working with Ec_Tentacle
	[*] Number of unique genes for celltype: 31
	[*] Remaining IDs for GO analysis: 12
[*] Working with I_FemGC
	[*] Number of unique genes for celltype: 38
	[*] Remaining IDs for GO analysis: 26
[*] Working with Ec_BasalDisk
	[*] Number of unique genes for celltype: 80
	[*] Remaining IDs for GO analysis: 29
[*] Working with I_Neuro
	[*] Number of unique genes for celltype: 3
	[*] Remaining IDs for GO analysis: 2
[*] Working

0

In [17]:
# conducting GO enrichment analysis with the find_enrichment.py script of GOATOOLS
# errors if sample files are empty e.g. I_ISC
for celltype in unique_celltypes.columns:
    if celltype != "ID":
        print("[*] Working with: {}".format(celltype))
        outfile = "../results/goatools_results/" + celltype + ".tsv"
        samplefile = "../data/goatools_data/sample_" + celltype + ".txt"
        !find_enrichment.py $samplefile ../data/goatools_data/population.txt ../data/goatools_data/associations.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile --obsolete replace > /dev/null
        print("\t[*] DONE")

[*] Working with: Ec_Head
	[*] DONE
[*] Working with: En_Head
	[*] DONE
[*] Working with: Ec_BodyCol.SC
	[*] DONE
[*] Working with: I_ISC
Traceback (most recent call last):
  File "/blast/miniconda3/bin/find_enrichment.py", line 44, in <module>
    main()
  File "/blast/miniconda3/bin/find_enrichment.py", line 31, in main
    obj = GoeaCliFnc(GoeaCliArgs().args)
  File "/blast/miniconda3/lib/python3.8/site-packages/goatools/cli/find_enrichment.py", line 362, in __init__
    self.chk_genes(_study, _pop, self.objanno.associations)
  File "/blast/miniconda3/lib/python3.8/site-packages/goatools/cli/find_enrichment.py", line 496, in chk_genes
    overlap = self.get_overlap(study, pop)
  File "/blast/miniconda3/lib/python3.8/site-packages/goatools/cli/find_enrichment.py", line 540, in get_overlap
    return float(len(study & pop)) / len(study)
ZeroDivisionError: float division by zero
	[*] DONE
[*] Working with: Ec_Peduncle
	[*] DONE
[*] Working with: Ec_Tentacle
	[*] DONE
[*] Working with: 

In [30]:
from matplotlib import font_manager
def plot_goa(goafile_enriched:pd.DataFrame,savep:str, celltype:str):
    print("[*] Producing plot for {}".format(celltype))
    goafile_enriched["ratio_stud"] = goafile_enriched.ratio_in_study.apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
    goafile_enriched["ratio_pop"] = goafile_enriched.ratio_in_pop.apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
    goafile_enriched["amount_in_pop"] = goafile_enriched.ratio_in_pop.apply(lambda x: int(x.split("/")[0]))

    categories = []
    for cat in list(goafile_enriched.name):
        if len(cat) >= 30:
            cat = textwrap.fill(cat, width=30)
        categories.append(cat)
        
    values = list(goafile_enriched.study_count)
    scatter_values = np.array(goafile_enriched.study_count) / np.array(goafile_enriched.amount_in_pop)
    
    pcolors = goafile_enriched.p_fdr_bh
    norm_p_values = np.array(pcolors) / max(pcolors)
    colors=plt.cm.RdBu_r(norm_p_values)

    # Create figure and axes
    if len(goafile_enriched) == 30:
        fsize = (20,18)
    elif len(goafile_enriched) >= 15:
        fsize = (16,12)
    else:
        fsize = (12,8)
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 16), sharey=True)

    # Plot horizontal bar plot on ax1
    ax1.barh(categories, values, color=colors, edgecolor="black")

    ax1.set_xlabel('Count', fontsize=15)
    font_props = font_manager.FontProperties(size=15)

    # Set the y-tick labels with the font properties
    ax1.set_yticklabels(categories, fontproperties=font_props)
    
    ax2.scatter(scatter_values, categories, c=colors, cmap='RdBu_r', 
                label='Gene Ratio (compared to Study)', s=list(goafile_enriched.ratio_stud*1000),edgecolor="black")

    ax2.set_xlabel('Count in Study / Count in Pop', fontsize=15)
    ax1.set_ylabel('GO Categories', fontsize=15)
    ax1.invert_yaxis()
    plt.subplots_adjust(left=0.2, wspace=0.1)
    cbar = fig.colorbar(ScalarMappable(cmap='RdBu_r'), ax=[ax1, ax2], pad = 0.005)
    cbar.set_label('p-values', fontsize=15)
    cbar.set_ticks([min(norm_p_values), max(norm_p_values)])
    cbar.set_ticklabels([f'{min(goafile_enriched.p_fdr_bh):.4f}', f'{max(goafile_enriched.p_fdr_bh):.4f}'])

    cbar.ax.set_position([0.85, 0.15, 0.03, 0.7])

    plt.savefig(savep + celltype + ".jpg", dpi=400)
    plt.close()
    print("[*] DONE")

In [31]:
savep = "../results/goatools_results/"
for celltype in unique_celltypes.columns:
    if celltype != "ID":
        filepath = "../results/goatools_results/" + celltype + ".tsv"
        if isfile(filepath):
            goafile = pd.read_table(filepath)
            goafile_enriched = goafile[goafile.enrichment == "e"]
            if len(goafile_enriched) > 50:
                bp_goafile_enriched = goafile_enriched[goafile_enriched.NS == "BP"]
                cc_goafile_enriched = goafile_enriched[goafile_enriched.NS == "CC"]
                mf_goafile_enriched = goafile_enriched[goafile_enriched.NS == "MF"]

                egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                go_categoriy = ["BP", "CC", "MF"]
                for goafile, process in zip(egoafiles, go_categoriy):
                    if len(goafile) > 30:
                        goafile = goafile.nsmallest(30, "p_fdr_bh")
                    plot_goa(goafile, savep=savep + process + "_", celltype=celltype)
            else:
                plot_goa(goafile_enriched, savep=savep, celltype=celltype)


[*] Producing plot for I_FemGC


/tmp/ipykernel_12/2765851321.py:38: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax1.set_yticklabels(categories, fontproperties=font_props)


[*] DONE
[*] Producing plot for I_MaleGC
[*] DONE
[*] Producing plot for I_ZymoGl


/blast/miniconda3/lib/python3.8/site-packages/pandas/core/frame.py:3607: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._set_item(key, value)


[*] DONE
[*] Producing plot for I_StenoNB
[*] DONE
[*] Producing plot for I_StenoNC
[*] DONE
[*] Producing plot for I_Ec1N
[*] DONE
[*] Producing plot for I_Ec4N
[*] DONE
[*] Producing plot for I_SpumMucGl
[*] DONE
[*] Producing plot for I_GranGl
[*] DONE
[*] Producing plot for I_Ec1.5N
[*] DONE
[*] Producing plot for C_neuron
[*] DONE
[*] Producing plot for C_neuron
[*] DONE
[*] Producing plot for C_neuron
[*] DONE
[*] Producing plot for C_germline
[*] DONE
[*] Producing plot for C_nematoblast
[*] DONE
[*] Producing plot for C_nematoblast
[*] DONE
[*] Producing plot for C_nematoblast
[*] DONE
[*] Producing plot for C_nematocytes
[*] DONE
[*] Producing plot for C_nematocytes
[*] DONE
[*] Producing plot for C_nematocytes
[*] DONE
[*] Producing plot for C_gland_cell
[*] DONE
[*] Producing plot for C_neuroprogenitor
[*] DONE
[*] Producing plot for C_EC_foot
[*] DONE
[*] Producing plot for C_EN_foot
[*] DONE
[*] Producing plot for C_tentacle
[*] DONE
[*] Producing plot for C_head
[*] DONE


In [32]:
def construct_study_files_for_transcriptomics(log2FoldChange_df:pd.DataFrame,population_table:pd.DataFrame,
                                              savep:str,result_path:str,experiment:str)->tuple:
    log2FoldChange_df["ID"] = np.array(log2FoldChange_df.index)
    log2FoldChange_df["ID"] = log2FoldChange_df["ID"].apply(lambda x: x.split(".")[1])
    
    log2FoldChange_df = log2FoldChange_df[log2FoldChange_df["ID"].isin(population_table["ID"])]
    log2FoldChange_df = log2FoldChange_df[log2FoldChange_df["padj"] <= 0.05]
    log2FoldChange_df_up = log2FoldChange_df[log2FoldChange_df["log2FoldChange"] >= 1]
    log2FoldChange_df_down = log2FoldChange_df[log2FoldChange_df["log2FoldChange"] <= -1]
    
    
    print("\t[*] Number of diff genes {}".format(len(log2FoldChange_df)))
    samplefile_up = savep+"sample_up_"+experiment+".txt"
    outfile_up = result_path + "sample_up_"+experiment+".tsv"
    
    
    with open(samplefile_up, "w") as samplefile:
        ident = []
        for identifier in list(log2FoldChange_df_up.ID):
            samplefile.write(identifier+"\n")
            ident.append(identifier)
        print("\t[*] Remaining IDs for Upregulated Genes GO analysis: {}".format(len(ident)))
    
    samplefile_down = savep+"sample_down_"+experiment+".txt"
    outfile_down = result_path + "sample_down_"+experiment+".tsv"
    with open(samplefile_down, "w") as samplefile:
        ident = []
        for identifier in list(log2FoldChange_df_down.ID):
            samplefile.write(identifier+"\n")
            ident.append(identifier)
    print("\t[*] Remaining IDs for Downregulated Genes GO analysis: {}".format(len(ident)))
    
    print("\t[*] Trying to perform find_enrichment analysis ...")
    try:
        print("\t[*] Analysing downregulated genes")
        !find_enrichment.py $samplefile_down ../data/goatools_data/population.txt ../data/goatools_data/associations.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_down --obsolete replace > /dev/null
    except Exception as e:
        print("\t[**] Warning: {}".format(e))
    try:
        print("\t[*] Analyzing upregulated genes")
        !find_enrichment.py $samplefile_up ../data/goatools_data/population.txt ../data/goatools_data/associations.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_up --obsolete replace > /dev/null
    except Exception as e:
        print("\t[**] Warning: {}".format(e))
    
    return outfile_up, outfile_down

In [33]:
pop = pd.read_table("../data/goatools_data/population.txt", header=None)
pop.columns = ["ID"]
pop.head()

,ID
0,G000002
1,G000009
2,G000010
3,G000012
4,G000022


In [34]:
filepath = "../results/deseq2_rsem/tables/"
experiments = listdir(filepath)
for exp in experiments:
    print("[*] Working on {}".format(exp))
    if "HydraRecolonization_Conventionalized" in exp or "HydraRecolonization_Cvbct" in exp:
        experiment_name = exp.split("_results")[0]
    else:
        experiment_name = exp.split("_vs_")[0]
    log2FoldChange_df = pd.read_csv(filepath + exp, header=0, index_col=0)
    outfile_up, outfile_down = construct_study_files_for_transcriptomics(log2FoldChange_df,pop,"../data/goatools_data/","../results/goatools_results/",experiment_name)
    if isfile(outfile_up):
        enriched_up = pd.read_table(outfile_up)
        enriched_up = enriched_up[enriched_up.enrichment == "e"]
        if len(enriched_up) > 50:
            bp_goafile_enriched = enriched_up[enriched_up.NS == "BP"]
            cc_goafile_enriched = enriched_up[enriched_up.NS == "CC"]
            mf_goafile_enriched = enriched_up[enriched_up.NS == "MF"]

            egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
            go_categoriy = ["BP", "CC", "MF"]
            for goafile, process in zip(egoafiles, go_categoriy):
                if len(goafile) > 30:
                    goafile = goafile.nsmallest(30, "p_fdr_bh")
                plot_goa(goafile, savep="../results/goatools_results/" + process + "_", celltype=experiment_name+"up")
        else:
            plot_goa(enriched_up, savep="../results/goatools_results/", celltype=experiment_name+"up")
    
    if isfile(outfile_down):
        enriched_down = pd.read_table(outfile_down)
        enriched_down = enriched_down[enriched_down.enrichment == "e"]
        if len(enriched_down) > 50:
            bp_goafile_enriched = enriched_down[enriched_down.NS == "BP"]
            cc_goafile_enriched = enriched_down[enriched_down.NS == "CC"]
            mf_goafile_enriched = enriched_down[enriched_down.NS == "MF"]

            egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
            go_categoriy = ["BP", "CC", "MF"]
            for goafile, process in zip(egoafiles, go_categoriy):
                if len(goafile) > 30:
                    goafile = goafile.nsmallest(30, "p_fdr_bh")
                plot_goa(goafile, savep="../results/goatools_results/" + process + "_", celltype=experiment_name+"down")
        else:
            plot_goa(enriched_down, savep="../results/goatools_results/", celltype=experiment_name+"down")


[*] Working on EcoKD1_Eco1KD_B5_vs_control_B5_results.csv
	[*] Number of diff genes 83
	[*] Remaining IDs for Upregulated Genes GO analysis: 5
	[*] Remaining IDs for Downregulated Genes GO analysis: 39
	[*] Trying to perform find_enrichment analysis ...
	[*] Analysing downregulated genes
	[*] Analyzing upregulated genes
[*] Producing plot for EcoKD1_Eco1KD_B5up


/tmp/ipykernel_12/2765851321.py:38: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax1.set_yticklabels(categories, fontproperties=font_props)


[*] DONE
[*] Producing plot for EcoKD1_Eco1KD_B5down
[*] DONE
[*] Working on EcoKD1_Eco1KD_B8_vs_control_B8_results.csv
	[*] Number of diff genes 299
	[*] Remaining IDs for Upregulated Genes GO analysis: 41
	[*] Remaining IDs for Downregulated Genes GO analysis: 38
	[*] Trying to perform find_enrichment analysis ...
	[*] Analysing downregulated genes
	[*] Analyzing upregulated genes
[*] Producing plot for EcoKD1_Eco1KD_B8down
[*] DONE
[*] Working on HydraAHL_3OC12_vs_control_results.csv
	[*] Number of diff genes 4508
	[*] Remaining IDs for Upregulated Genes GO analysis: 443
	[*] Remaining IDs for Downregulated Genes GO analysis: 353
	[*] Trying to perform find_enrichment analysis ...
	[*] Analysing downregulated genes
	[*] Analyzing upregulated genes
[*] Producing plot for HydraAHL_3OC12up
[*] DONE
[*] Producing plot for HydraAHL_3OC12up
[*] DONE
[*] Producing plot for HydraAHL_3OC12up
[*] DONE
[*] Producing plot for HydraAHL_3OC12down
[*] DONE
[*] Working on HydraAHL_3OHC12_vs_control

/blast/miniconda3/lib/python3.8/site-packages/pandas/core/frame.py:3607: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._set_item(key, value)


[*] DONE
[*] Producing plot for HydraRecolonization_Conventionalized_vs_GFup
[*] DONE
[*] Producing plot for HydraRecolonization_Conventionalized_vs_GFup
[*] DONE
[*] Producing plot for HydraRecolonization_Conventionalized_vs_GFdown
[*] DONE
[*] Working on HydraRecolonization_Conventionalized_vs_Wild_results.csv
	[*] Number of diff genes 50
	[*] Remaining IDs for Upregulated Genes GO analysis: 8
	[*] Remaining IDs for Downregulated Genes GO analysis: 0
	[*] Trying to perform find_enrichment analysis ...
	[*] Analysing downregulated genes
Traceback (most recent call last):
  File "/blast/miniconda3/bin/find_enrichment.py", line 44, in <module>
    main()
  File "/blast/miniconda3/bin/find_enrichment.py", line 31, in main
    obj = GoeaCliFnc(GoeaCliArgs().args)
  File "/blast/miniconda3/lib/python3.8/site-packages/goatools/cli/find_enrichment.py", line 362, in __init__
    self.chk_genes(_study, _pop, self.objanno.associations)
  File "/blast/miniconda3/lib/python3.8/site-packages/goato

In [22]:
# idea compare two transcriptomics dataframes for specific celltypes ..

# KEGG Enrichment Analysis

In [23]:
unique_celltypes = pd.read_table("../results/processedData/normalized_mean/unique_celltypes.table", sep=";")

In [24]:
merged_df = annotation_df.merge(unique_celltypes, how='inner', on='ID')[annotation_df.columns]
merged_df.head()

,ID,KO,KO_definition,Score,Second.best,Score2,UniProtKB,KEGG.ID,KO.Number,GO.terms
0,NaN,K25587,E3.4.17.25; glutathione-S-conjugate glycine hy...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,K25587,E3.4.17.25; glutathione-S-conjugate glycine hy...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,K25587,E3.4.17.25; glutathione-S-conjugate glycine hy...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,K25587,E3.4.17.25; glutathione-S-conjugate glycine hy...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,K25587,E3.4.17.25; glutathione-S-conjugate glycine hy...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
cleaned_merged_df = merged_df[["ID","KO"]].dropna()
cleaned_merged_df = cleaned_merged_df.drop_duplicates()
cleaned_merged_df.to_csv("../results/gene_enrichment/gene_to_KO.csv")

In [26]:
unique_celltypes = pd.read_table("../results/processedData/normalized_mean/unique_celltypes.table", sep=";")
unique_celltypes = unique_celltypes.fillna(0) 

ecokd1 = pd.read_csv("../results/deseq2_rsem/tables/EcoKD1_Eco1KD_B8_vs_control_B8_results.csv", header=0, index_col=0)
wild = pd.read_csv("../results/deseq2_rsem/tables/HydraRecolonization_Wild_vs_GF_results.csv", header=0, index_col=0)
cvbct = pd.read_csv("../results/deseq2_rsem/tables/HydraRecolonization_Cvbct_vs_GF_results.csv",header=0, index_col=0)
temp = pd.read_csv("../results/deseq2_rsem/tables/HydraTemperature_08°C_vs_18°C_results.csv",header=0, index_col=0)

In [27]:
# returns a dataframe with log2foldchanges for the relevant celltype and transcriptome experiment
def get_unique_genes_diffexpression_table(transcriptome_df:pd.DataFrame,celltype_df:pd.DataFrame,celltype:str)->pd.DataFrame:
    try:
        unique_genes = list(celltype_df[celltype_df[celltype] == 1].ID.apply(lambda x: x.split("-")[1]))
        transcriptome_df['gene_id'] = transcriptome_df.index
        transcriptome_df['gene_id'] = transcriptome_df['gene_id'].apply(lambda x: x.split(".")[1])
        #transcriptome_df = transcriptome_df[transcriptome_df["padj"] <= 0.05]
        #transcriptome_df = transcriptome_df[abs(transcriptome_df["log2FoldChange"]) >= 1]
        return transcriptome_df[transcriptome_df['gene_id'].isin(unique_genes)]
    except Exception as e:
        raise Exception("[-] ERROR with exception: {}".format(e))
        

def extract_celltype_genes(in_celltypes:pd.DataFrame,experiment_dataframe:pd.DataFrame,celltype:str)->pd.DataFrame:
    celltypes = in_celltypes.copy()
    cleaned_celltypes = celltypes[celltypes[celltype].isna() == False][["ID",celltype]]
    experiment_dataframe_copy = experiment_dataframe.copy()
    experiment_dataframe_copy["ID"] = experiment_dataframe.index
    experiment_dataframe_copy["ID"] = experiment_dataframe_copy.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1])
    #experiment_dataframe_copy = experiment_dataframe_copy[experiment_dataframe_copy['padj'] <= 0.05]
    #experiment_dataframe_copy = experiment_dataframe_copy[abs(experiment_dataframe_copy['log2FoldChange']) >= 1]
    
    experiment_celltype = pd.merge(experiment_dataframe_copy, cleaned_celltypes, on="ID")[["ID","log2FoldChange","pvalue",celltype]]
    experiment_celltype = experiment_celltype[experiment_celltype[celltype] != 0]

    print(len(experiment_celltype), celltype)
    return experiment_celltype

In [28]:
def produce_tables_for_all_celtypes(dataframe:pd.DataFrame,
                                    celltypes:pd.DataFrame,
                                    id_to_ko_dataframe:pd.DataFrame,experiment:str):
    try:
        print("[*] working on {}".format(experiment))
        for celltype in unique_celltypes.columns:
            if celltype != "ID":
                print("[*] working with: {}".format(celltype))
                celltype_dataframe = get_unique_genes_diffexpression_table(dataframe, unique_celltypes, celltype)
                celltype_dataframe["ID"] = list(celltype_dataframe.index)
                celltype_dataframe["ID"] = celltype_dataframe.ID.apply(lambda x: x.split(".")[1])
                celltype_dataframe = id_to_ko_dataframe.merge(celltype_dataframe, on="ID")
                print("\t[*] Number of Diff. Genes: {}".format(len(celltype_dataframe)))
                if isdir("../results/gene_enrichment/" + experiment + "/") == False:
                    mkdir("../results/gene_enrichment/" + experiment + "/")
                    
                savep = "../results/gene_enrichment/" + experiment + "/" + experiment + "_" + celltype + ".csv"
                
                celltype_dataframe[["ID","KO"]].to_csv(savep)
                
        print("[+] DONE")
        return 0
    except Exception as e:
        raise Exception(" [-] ERROR with exception: {}".format(e))

In [29]:
produce_tables_for_all_celtypes(ecokd1, unique_celltypes, cleaned_merged_df, "ecokd1")

[*] working on ecokd1
[*] working with: Ec_Head
	[*] Number of Diff. Genes: 0
[*] working with: En_Head
	[*] Number of Diff. Genes: 0
[*] working with: Ec_BodyCol.SC
	[*] Number of Diff. Genes: 0
[*] working with: I_ISC
	[*] Number of Diff. Genes: 0
[*] working with: Ec_Peduncle
	[*] Number of Diff. Genes: 0
[*] working with: Ec_Tentacle
	[*] Number of Diff. Genes: 0
[*] working with: I_FemGC
	[*] Number of Diff. Genes: 0
[*] working with: Ec_BasalDisk
	[*] Number of Diff. Genes: 0
[*] working with: I_Neuro
	[*] Number of Diff. Genes: 0
[*] working with: I_EarlyNem
	[*] Number of Diff. Genes: 0
[*] working with: I_MaleGC
	[*] Number of Diff. Genes: 0
[*] working with: I_En2N
	[*] Number of Diff. Genes: 0
[*] working with: En_Foot
	[*] Number of Diff. Genes: 0
[*] working with: En_BodyCol.SC
	[*] Number of Diff. Genes: 0
[*] working with: I_ZymoGl
	[*] Number of Diff. Genes: 0
[*] working with: I_GlProgen
	[*] Number of Diff. Genes: 0
[*] working with: I_StenoNB
	[*] Number of Diff. Gen

/blast/miniconda3/lib/python3.8/site-packages/pandas/core/frame.py:3607: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._set_item(key, value)


[*] working with: I_GranGl
	[*] Number of Diff. Genes: 0
[*] working with: I_DesmoNB
	[*] Number of Diff. Genes: 0
[*] working with: I_IsoNB
	[*] Number of Diff. Genes: 0
[*] working with: I_Ec2N
	[*] Number of Diff. Genes: 0
[*] working with: I_Ec1.5N
	[*] Number of Diff. Genes: 0
[*] working with: I_En1N
	[*] Number of Diff. Genes: 0
[*] working with: I_En3N
	[*] Number of Diff. Genes: 0
[*] working with: C_neuron
	[*] Number of Diff. Genes: 0
[*] working with: C_germline
	[*] Number of Diff. Genes: 0
[*] working with: C_nematoblast
	[*] Number of Diff. Genes: 0
[*] working with: C_nematocytes
	[*] Number of Diff. Genes: 0
[*] working with: C_gland_cell
	[*] Number of Diff. Genes: 0
[*] working with: C_progenitor
	[*] Number of Diff. Genes: 0
[*] working with: C_neuroprogenitor
	[*] Number of Diff. Genes: 0
[*] working with: C_EC_foot
	[*] Number of Diff. Genes: 0
[*] working with: C_EN_foot
	[*] Number of Diff. Genes: 0
[*] working with: C_tentacle
	[*] Number of Diff. Genes: 0
[*] 

0

In [30]:
produce_tables_for_all_celtypes(wild, unique_celltypes, cleaned_merged_df, "wild")

[*] working on wild
[*] working with: Ec_Head
	[*] Number of Diff. Genes: 0
[*] working with: En_Head
	[*] Number of Diff. Genes: 0
[*] working with: Ec_BodyCol.SC
	[*] Number of Diff. Genes: 0
[*] working with: I_ISC
	[*] Number of Diff. Genes: 0
[*] working with: Ec_Peduncle
	[*] Number of Diff. Genes: 0
[*] working with: Ec_Tentacle
	[*] Number of Diff. Genes: 0
[*] working with: I_FemGC
	[*] Number of Diff. Genes: 0
[*] working with: Ec_BasalDisk
	[*] Number of Diff. Genes: 0
[*] working with: I_Neuro
	[*] Number of Diff. Genes: 0
[*] working with: I_EarlyNem
	[*] Number of Diff. Genes: 0
[*] working with: I_MaleGC
	[*] Number of Diff. Genes: 0
[*] working with: I_En2N
	[*] Number of Diff. Genes: 0
[*] working with: En_Foot
	[*] Number of Diff. Genes: 0
[*] working with: En_BodyCol.SC
	[*] Number of Diff. Genes: 0
[*] working with: I_ZymoGl
	[*] Number of Diff. Genes: 0
[*] working with: I_GlProgen
	[*] Number of Diff. Genes: 0
[*] working with: I_StenoNB
	[*] Number of Diff. Genes

0

In [31]:
produce_tables_for_all_celtypes(temp, unique_celltypes, cleaned_merged_df, "temp")

[*] working on temp
[*] working with: Ec_Head
	[*] Number of Diff. Genes: 0
[*] working with: En_Head
	[*] Number of Diff. Genes: 0
[*] working with: Ec_BodyCol.SC
	[*] Number of Diff. Genes: 0
[*] working with: I_ISC
	[*] Number of Diff. Genes: 0
[*] working with: Ec_Peduncle
	[*] Number of Diff. Genes: 0
[*] working with: Ec_Tentacle
	[*] Number of Diff. Genes: 0
[*] working with: I_FemGC
	[*] Number of Diff. Genes: 0
[*] working with: Ec_BasalDisk
	[*] Number of Diff. Genes: 0
[*] working with: I_Neuro
	[*] Number of Diff. Genes: 0
[*] working with: I_EarlyNem
	[*] Number of Diff. Genes: 0
[*] working with: I_MaleGC
	[*] Number of Diff. Genes: 0
[*] working with: I_En2N
	[*] Number of Diff. Genes: 0
[*] working with: En_Foot
	[*] Number of Diff. Genes: 0
[*] working with: En_BodyCol.SC
	[*] Number of Diff. Genes: 0
[*] working with: I_ZymoGl
	[*] Number of Diff. Genes: 0
[*] working with: I_GlProgen
	[*] Number of Diff. Genes: 0
[*] working with: I_StenoNB
	[*] Number of Diff. Genes

0

In [32]:
ecokd_male_germline = get_unique_genes_diffexpression_table(ecokd1, unique_celltypes, 'I_MaleGC')
wild_male_germline = get_unique_genes_diffexpression_table(wild, unique_celltypes, 'I_MaleGC')
temp_male_germline = get_unique_genes_diffexpression_table(temp, unique_celltypes, 'I_MaleGC')

ecokd_female_germline = get_unique_genes_diffexpression_table(ecokd1, unique_celltypes, 'I_FemGC')
wild_female_germline = get_unique_genes_diffexpression_table(wild, unique_celltypes, 'I_FemGC')
temp_female_germline = get_unique_genes_diffexpression_table(temp, unique_celltypes, 'I_FemGC')

In [33]:
ecokd_male_germline["ID"] = list(ecokd_male_germline.index)
ecokd_male_germline["ID"] = ecokd_male_germline.ID.apply(lambda x: x.split(".")[1])

ecokd1_merged_df = cleaned_merged_df.merge(ecokd_male_germline, on="ID")
ecokd1_merged_df[["ID","KO"]].to_csv("../results/gene_enrichment/ecokd1_malegermline.csv")